# SpkIOSzero — Sparking! ZERO para iOS

Pipeline isolado da branch `SpkIOSzero`, seguindo a arquitetura Route C do Link2iOS. A entrada do jogo vem exclusivamente de `Drive/<pasta configurada>/spk`; update, DLCs e `prod.keys` não são procurados nos diretórios do BOTW. O Colab prepara/extrai e envia o workspace; o Actions macOS faz Unreal/Oodle, shader split, Metal, framework e Shared Shell.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from pathlib import Path
import sys, subprocess

REPO = 'joao9vegeto-stack/KintTany'
BRANCH = 'SpkIOSzero'
REPO_DIR = Path('/content/KintTany-SpkIOSzero')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    f'https://github.com/{REPO}.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
sys.path.insert(0, str(REPO_DIR / 'colab'))
import run_spk

# Mesmo ID da pasta compartilhada já configurada no pipeline Link2iOS/Kleo2iOS.
DRIVE_FOLDER_ID = '19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm'
api = run_spk.drive_api()
args = type('Args', (), {'drive_folder_id': DRIVE_FOLDER_ID})()
print('Repositório:', REPO, '| branch:', BRANCH)
print('Pasta de origem do Drive:', DRIVE_FOLDER_ID)

In [ ]:
from getpass import getpass
import os

# Primeiro tenta recuperar uma preparação SPK completa já enviada ao Drive.
handoff = run_spk.recover(args, api)
if handoff is None:
    root_items = run_spk.list_children(api, DRIVE_FOLDER_ID)
    folder_mime = 'application/vnd.google-apps.folder'
    spk_folder = next((x for x in root_items
        if x.get('mimeType') == folder_mime and x.get('name', '').lower() == 'spk'), None)
    if spk_folder is None:
        raise RuntimeError("A pasta configurada do Drive precisa conter a subpasta 'spk'.")
    spk_items = run_spk.list_children(api, spk_folder['id'])
    package_files = [x for x in spk_items
        if x.get('mimeType') != folder_mime and x.get('name', '').lower().endswith('.nsp')]
    base_candidates = [x for x in package_files
        if x.get('name', '').lower() != 'update.nsp'
        and not x.get('name', '').lower().startswith('dlc')]
    if len(base_candidates) == 0:
        # A URL assinada do 7z é solicitada sem eco e nunca é gravada no repositório/Drive.
        os.environ['SPK_BASE_ARCHIVE_URL'] = getpass(
            'Cole o link assinado atualizado do 7z do jogo base (URL temporária): '
        ).strip()
    elif len(base_candidates) == 1:
        os.environ['SPK_BASE_ARCHIVE_URL'] = ''
        print('Base NSP encontrada em Drive/spk:', base_candidates[0]['name'], '— não será baixado o 7z.')
    else:
        raise RuntimeError('Drive/spk tem mais de um NSP candidato a jogo base. Deixe apenas um além de update.nsp e dlc*.nsp.')

    handoff = run_spk.prepare(args, api)
else:
    print('Preparação SPK recuperada do Drive; não repetindo download, extração, rewrite ou upload.')

print('Handoff pronto:', handoff['request_id'])
print('Build:', handoff['package_version'])
print('Pasta de saída:', handoff['drive_output_folder_url'])

In [ ]:
# Commit handoff.json na branch SpkIOSzero. O commit dispara apenas o workflow desta branch.
handoff_path = Path('/content/spk-ios-zero-handoff.json')
if not handoff_path.is_file():
    handoff_path.write_text(__import__('json').dumps(handoff, indent=2) + '\n', encoding='utf-8')
dispatch_args = type('Args', (), {'handoff': handoff_path})()
run_spk.dispatch(dispatch_args)

## Depois do dispatch

O GitHub Actions valida o handoff na branch `SpkIOSzero`, restaura apenas o workspace e os dados externos do jogo, executa Unreal/Oodle e o restante da Route C no runner `xcode-27`. O IPA e o `build-report.json` ficam nos artifacts do workflow; os logs são enviados em um artifact separado.